In [141]:
import torch
from cpu_cb import E8P12_codebook

In [142]:
cliques = [
    [0, 2, 11, 25, 33, 39, 47, 57],
    [1, 3, 10, 24, 32, 38, 46, 56],
    [4, 6, 15, 29, 35, 37, 43, 61],
    [5, 7, 14, 28, 34, 36, 42, 60],
    [12, 18, 20, 26, 44, 53, 55, 62],
    [13, 19, 21, 27, 45, 52, 54, 63],
    [8, 16, 22, 30, 40, 49, 51, 58],
    [9, 17, 23, 31, 41, 48, 50, 59]
]
cperm = torch.tensor(cliques).view(-1)
ciperm = cperm.sort().indices # gets original indices of elements after sorted

In [143]:
def hbc_transform(x):
    mm = x.shape[:-1]
    x = x.view(-1, x.shape[-1])
    m, n = x.shape
    k = 1
    while 4**k < n: k += 1
    assert 4**k == n
    b = torch.tensor([1,1,-1,-1], dtype=x.dtype, device=x.device)
    x = x.reshape(-1,64)[:,ciperm]
    x = x.reshape((m,) + (4,)*k)
    for i in range(k):
        x = x.flip(1+i) + x * b.view((1,)*(i+1)+(4,)+(1,)*(k-1-i))
    x = x.reshape((m,) + (2,2)*k)
    x = x.permute((0,)+tuple(2*i+1 for i in range(k))+tuple(2*i+2 for i in range(k)))
    return x.reshape(mm + (2**k, 2**k)) / (2**(k/2))
# ihbc_transform(hbc_transform(x)) = x
def ihbc_transform(x):
    mm = x.shape[:-2]
    x = x.view(-1, x.shape[-2], x.shape[-1])
    m, n, n_ = x.shape
    assert(n == n_)
    k = 1
    while 2**k < n: k += 1
    assert 2**k == n
    x = x.reshape((m,) + (2,2)*k)
    x = x.permute((0,)+tuple((i>>1)+1+(i&1)*k for i in range(2*k)))
    b = torch.tensor([1,1,-1,-1], dtype=x.dtype, device=x.device)
    x = x.reshape((m,) + (4,)*k)
    for i in range(k):
        x = x.flip(1+i) + x * b.view((1,)*(i+1)+(4,)+(1,)*(k-1-i))
    x = x.reshape(-1,64)[:,cperm]
    return x.reshape(mm + (4**k,)) / (2**(k/2))

In [144]:
# just multiply by H lifted into the higher space (lift H into the coefficient space)
def H_multiply(X,H):
    X = hbc_transform(X)
    X = X @ H
    X = ihbc_transform(X)
    return X

In [145]:
# the multiplication by lifted H masked by the lower triangular mask
def HL_multiply(X,H):
    Xshape = X.shape
    n = X.shape[-1]
    Z = X.clone()
    Z = Z.reshape(-1,n)
    Y = torch.zeros_like(Z)
    for i in range(n):
        Y[:,i] = H_multiply(Z,H)[:,i]
        Z[:,i].zero_()
    return Y.reshape(Xshape)

In [146]:
def decompose_H(H):
    (n, n_) = H.shape
    assert(n_ == n)
    decomp = []
    while n > 8:
        next_n = n // 2
        Ha, Hb = H[:next_n, :next_n], H[:next_n, next_n:]
        Hc, Hd = H[next_n:, :next_n], H[next_n:, next_n:]
        H_decomp = [(Ha+Hd)/2, (Hb+Hc)/2, (Hb-Hc)/2, (Ha-Hd)/2]
        decomp.append(H_decomp)
        H = H_decomp[0]
        n = next_n
    assert(n == 8)
    # lift H
    X = hbc_transform(torch.eye(64))
    X = X @ H
    X = ihbc_transform(X)
    decomp.append(torch.tril(X)) # J operator for base case, 64x64
    return decomp

In [ ]:
def fast_HL_multiply(X,H):
    return fast_HL_multiply_sub(X,decompose_H(H))

def fast_HL_multiply_sub(X,DEC):
    if len(DEC) == 1:
        (Y,) = DEC
        return X @ Y
    (H0,H1,H2,H3) = DEC[0]
    Xshape = X.shape
    n = X.shape[-1]
    X = X.reshape(-1,4,n//4)
    Y = fast_HL_multiply_sub(X,DEC[1:])
    Y[:,0,:] += H_multiply(X[:,1,:],H1) + H_multiply(X[:,3,:],H3) - H_multiply(X[:,2,:],H2)
    Y[:,1,:] += H_multiply(X[:,3,:],H2) - H_multiply(X[:,2,:],H3)
    Y[:,2,:] += H_multiply(X[:,3,:],H1)
    return Y.reshape(Xshape)

In [148]:
cb = E8P12_codebook()

In [149]:
# def bulk_LDLQ(X,A,cb,eps=1e-5):
#     A = A / A.diag().mean()
#     Xscale = X.square().mean().sqrt() / cb.opt_scale
#     X = X / Xscale
#     DEC = decompose_H(A)
#     hatX = cb.quantize(X.reshape(-1,8))[0].reshape(X.shape)
#     for ii in range(X.shape[-1]):
#         Y = hatX + fast_HL_multiply_sub(X - hatX, DEC)
#         hatX_next = cb.quantize(Y.reshape(-1,8))[0].reshape(X.shape)
#         if (hatX_next - hatX).square().mean() <= eps * hatX.square().mean():
#             print(f'converged after {ii+1} steps')
#             break
#         hatX = hatX_next
#     return hatX * Xscale
def bulk_LDLQ(X,A,cb,eps=1e-5):
    A = A / A.diag().mean()
    Xscale = X.square().mean().sqrt() / cb.opt_scale
    X = X / Xscale
    DEC = decompose_H(A)
    hatX = cb.quantize(X.reshape(-1,8))[0].reshape(X.shape)
    Y = hatX + fast_HL_multiply_sub(X - hatX, DEC)
    hatX_next = cb.quantize(Y.reshape(-1,8))[0].reshape(X.shape)
    return hatX_next * Xscale

In [ ]:
# synthetic H
n = 64
H = torch.randn(n,n)
H = H @ H.t()
H = (H + H.t())/2
H.diagonal().add_(H.diagonal().mean() * 0.01)
S, V = torch.linalg.eigh(H)
S_sqrt = torch.sqrt(torch.clamp(S, min=0))
H_sqrt = V @ torch.diag(S_sqrt) @ V.T
W = torch.randn(1,n**2)
coeffs = ihbc_transform(W.reshape(1, n, n))

In [151]:
# adaptive rounding
hatW_coeff_ar = bulk_LDLQ(coeffs, H_sqrt, cb)
hatW_ar = hbc_transform(hatW_coeff_ar).reshape(1, n**2)
error = (W - hatW_ar).norm() / W.norm()
print(f"Error: {error}")
E = (hatW_ar - W).reshape(n, n)
proxy_loss = (E @ H @ E.T).trace()
print(f"Proxy loss: {proxy_loss}")

Error: 0.3212166428565979
Proxy loss: 21138.830078125


In [152]:
# non-adaptive rounding
# Z = cb.quantize(X.reshape(-1,8))[0].reshape(X.shape)
hatW_coeff_nr = bulk_LDLQ(coeffs, torch.eye(n), cb)
hatW_nr = hbc_transform(hatW_coeff_nr).reshape(1, n**2)
error = (W - hatW_nr).norm() / W.norm()
print(f"Error: {error}")
E = (hatW_nr - W).reshape(n, n)
proxy_loss = (E @ H @ E.T).trace()
print(f"Proxy loss: {proxy_loss}")

Error: 0.29857075214385986
Proxy loss: 22854.60546875
